In [ ]:
"""
Herbarium Processor Notebook
========================

This notebook is designed to process herbarium specimen images and generate labels using a prompt-based approach.
It includes steps for image processing, prompt configuration, and generating specimen labels.
It is structured to be run in a Jupyter notebook environment.

Configuration: select what images to sample from.

You should edit this section each time you run the code to:
  * select the images you want to process
  * set the path to the output CSV file.
"""

from utils.notebook_setup import setup_project

setup_project()

image_path = "data/img/bucket"
output_csv_path = "tmp/an_output.csv"

Notebook root set to /Users/danielleward/Documents/Projects/parsely_studio


In [16]:
"""
Configuration: setting up the prompts and system instructions for Gemini inference.

You do NOT need to edit this section.
"""

from herbarium_processor.core.inference.prompt_factory import (
    create_prompt_builder_from_yaml,
)

system_instructions_path = "prompts/ocr_system_instructions_no_citations.md"
prompt_builder = create_prompt_builder_from_yaml("prompts/configs/default_prompt.yaml")

print("Prompt:")
print(prompt_builder.generate_debug_prompt())

Prompt:
You are an AI assistant trained to extract structured data from herbarium specimen label images.

You will be shown 3 label images and corresponding OCR blocks as examples. These examples include the correct structured JSON output. After that, you will be shown a fourth image (with OCR blocks) that you must process.

Use both the image and the OCR output to extract only the following 21 fields as a valid JSON object using `snake_case` keys:

* `catalogNumber`
* `datasetName`
* `scientificName`
* `locality`
* `municipality`
* `county`
* `stateProvince`
* `country`
* `verbatimLatitude`
* `verbatimLongitude`
* `verbatimElevation`
* `habitat`
* `verbatimEventDate`
* `eventDate`
* `recordedBy`
* `recordNumber`
* `identifiedBy`
* `dateIdentified`
* `otherCatalogNumbers`
* `associatedTaxa`
* `comment`


Return `null` for any field that is not clearly present or legible. Do **not** infer or fabricate any information. Do **not** return extra fields.

---
**Example 1**

Image:
<|image_0|

In [17]:
"""
Step 1: Image processing.

Take the .heic images from the phone and convert them to .jpg format.
This step also includes resizing the images to a standard size and doing any necessary preprocessing.

You do NOT need to edit this section.
"""

from herbarium_processor.core.image.image_utils import process_directory_no_auto_rotate

process_directory_no_auto_rotate(image_path)

[970977.jpg] Original size: (1000, 751)
[970977.jpg] No resize needed.
[970977.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/970977.jpg
[1074331.jpg] Original size: (800, 549)
[1074331.jpg] No resize needed.
[1074331.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/1074331.jpg
[970976.jpg] Original size: (1000, 713)
[970976.jpg] No resize needed.
[970976.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/970976.jpg
[1008229.jpg] Original size: (1000, 684)
[1008229.jpg] No resize needed.
[1008229.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/1008229.jpg
[970967.jpg] Original size: (1000, 445)
[970967.jpg] No resize needed.
[970967.jpg] Saved corrected image to /Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/970967.jpg
[IMG_2716.jpg] Original size: (999, 750)
[I

In [11]:
"""
Step 2: Calling the OCR AI model.
This step will call OCR and save the output to .json files in the tmp/ directory.

It will then build specimen labels based on the OCR output.
You do NOT need to edit this section.
"""

import os

from herbarium_processor.core.ocr.ocr_client import OcrClient

ocr = OcrClient()

if image_path:
    for filename in os.listdir(image_path):
        path = os.path.join(image_path, filename)
        if filename.lower().endswith(".jpg") or filename.lower().endswith(".jpeg"):
            print(f"Processing {path}")
            ocr.extract_text_json(path)

Processing data/img/bucket/970977.jpg
Saved visualized image with bounding boxes and sources to /Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_bounding_970977.jpg
[{'id': '1.1.1', 'text': 'The New York Botanical Garden', 'bounding_box': [{'x': 282, 'y': 79}, {'x': 685, 'y': 79}, {'x': 685, 'y': 103}, {'x': 282, 'y': 103}], 'average_confidence': 0.98818898}, {'id': '1.1.2', 'text': 'LICHENS OF TENNESSEE , U.S.A.', 'bounding_box': [{'x': 257, 'y': 113}, {'x': 710, 'y': 113}, {'x': 710, 'y': 135}, {'x': 257, 'y': 135}], 'average_confidence': 0.9735189739999999}, {'id': '1.1.3', 'text': 'Coccocarpia palmicola ( Sprengel ) Arvidsson &', 'bounding_box': [{'x': 187, 'y': 185}, {'x': 813, 'y': 185}, {'x': 813, 'y': 220}, {'x': 187, 'y': 220}], 'average_confidence': 0.9648604728571428}, {'id': '1.1.4', 'text': 'Galloway', 'bounding_box': [{'x': 669, 'y': 220}, {'x': 784, 'y': 220}, {'x': 784, 'y': 246}, {'x': 669, 'y': 246}], 'average_confidence': 0.9862027}, {'id': '1.1.5', 'tex

In [18]:
"""
Step 3: Building the samples for the prompt based on the OCR output.

This section will automatically generate the list of specimen labels
based on the sampled paths defined above.

You do NOT need to edit this section.
"""

from pathlib import Path

from herbarium_processor.core.types.specimen_label import SpecimenLabel

target_examples = []

if image_path:
    for filename in os.listdir(image_path):
        if filename.startswith(".DS"):
            continue
        path = Path(os.path.join(image_path, filename))
        if path.suffix.lower() not in {".jpg", ".jpeg"}:
            continue
        ocr_file = Path("tmp") / f"ocr_ai_input_{path.stem}.json"
        if not ocr_file.exists():
            print(f"Skipping {filename}, OCR file not found: {ocr_file}")
            continue
        label = SpecimenLabel(
            id=path.stem,
            img_path=str(path),
            ocr_path=f"tmp/ocr_ai_input_{path.stem}.json",
        )
        target_examples.append(label)
        print("Saving specimen label with the following info:" + str(label))

Saving specimen label with the following info:SpecimenLabel(id='970977', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/970977.jpg'), ocr_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_ai_input_970977.json'))
Saving specimen label with the following info:SpecimenLabel(id='1074331', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/1074331.jpg'), ocr_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_ai_input_1074331.json'))
Saving specimen label with the following info:SpecimenLabel(id='970976', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/data/img/bucket/970976.jpg'), ocr_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/tmp/ocr_ai_input_970976.json'))
Saving specimen label with the following info:SpecimenLabel(id='1008229', img_path=PosixPath('/Users/danielleward/Documents/Projects/parsely_studio/

In [19]:
"""
Step 4: Calling the AI model

You do NOT need to edit this section.

"""

from herbarium_processor.core.inference.label_extraction_batch_runner import (
    LabelExtractionBatchRunner,
)

runner = LabelExtractionBatchRunner(
    output_csv_path=output_csv_path,
    system_instructions_path=system_instructions_path,
    prompt_builder=prompt_builder,
    targets=target_examples,
)
await runner.run_async()

[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
[DEBUG] Sending request to LLM at  https://openrouter.ai/api/v1  with model  google/gemini-2.5-pro
Saved outp

CancelledError: 